### data collection and processing
To aquire the necessary song lyric data, I am using [lyrics.ovh's API](https://github.com/NTag/lyrics.ovh). This is an easy process when using pythons request module. The API then returns a json string with a lyrics keyword, which I am adding to the already existing keys 'title' and 'artist'.

In [146]:
import re
import json
import string
import api_key
import requests
import lyricsgenius

In [147]:
# open the json-file containing all songs categorized by year and keyed with artist and title
with open('test.json', 'r') as file:
    data = json.load(file)

In [148]:
# write an empty json file for all songs to be analized
with open('songlist_with_lyrics.json', 'w') as f:
    json.dump({'songs':[]}, f, indent=4)

In [149]:
# function to append the json-string containing the lyrics to the already existing entries
def write_to_file(per_song_data):
    with open('songlist_with_lyrics.json', 'r') as f:
        file_data = json.load(f)
        file_data['songs'].append(per_song_data)

    with open('songlist_with_lyrics.json', 'w') as f:
        json.dump(file_data, f, indent=4)

In [150]:
# strip linebreaks '\n', punctuation and wrong words (like part names of songs) from song text provided by the API
def process_lyrics(lyrics):
    processed_lyrics = lyrics.lower()
    processed_lyrics = re.sub('[\[].*?[\]]', '', processed_lyrics)

    translation_table_punct = str.maketrans('', '', string.punctuation)
    translation_table_linebreaks = str.maketrans('\n', ' ')
    processed_lyrics = processed_lyrics.translate(translation_table_punct).translate(translation_table_linebreaks)

    processed_lyrics = processed_lyrics.lstrip()
    #processed_lyrics = processed_lyrics.replace('   ', ' ')
    return processed_lyrics

<>:4: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<>:4: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
/tmp/ipykernel_13608/464409094.py:4: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
  processed_lyrics = re.sub('[\[].*?[\]]', '', processed_lyrics)


In [151]:
genius_api = lyricsgenius.Genius(api_key.client_access_token)

In [152]:
# looping through the songs per year
for year in data:
    for song in range(len(data[year])):
        artist = data[year][song]['artist']
        title = data[year][song]['title']

        # append year to song's existing data
        year_dict = {'year': f"{year}"}
        data_per_song = data[year][song]
        data_per_song.update(year_dict)

        # API call, queried by artist and title of each song
        lyrics = genius_api.search_song(artist, title).lyrics
        
        # clean up the data from the API and update the content of the json-string
        lyrics = process_lyrics(lyrics)

        # append the lyrics to the song's existing data
        lyrics_dict = {'lyrics': f"{lyrics}"}
        data_per_song.update(lyrics_dict)

        # write to 'songlist_with_lyrics.json'
        write_to_file(data_per_song)

Timeout: Request timed out:
HTTPSConnectionPool(host='api.genius.com', port=443): Read timed out. (read timeout=5)

In [ ]:
# # looping through the songs per year
# for year in data:
#     for song in range(len(data[year])):
#         artist = data[year][song]['artist']
#         title = data[year][song]['title']

#         # append year to song's existing data
#         year_dict = {'year': f"{year}"}
#         data_per_song = data[year][song]
#         data_per_song.update(year_dict)

#         # API call, queried by artist and title of each song
#         url = f"https://api.lyrics.ovh/v1/{artist}/{title}"
    
#         response = requests.get(url)
#         lyrics = response.json()

#         # do not add song, if no lyrics could be fetched
#         if list(lyrics)[0] == 'error':
#             continue

#         # clean up the data from the API and update the content of the json-string
#         lyrics['lyrics'] = process_lyrics(lyrics['lyrics'])

#         # append the lyrics to the song's existing data
#         data_per_song.update(lyrics)

#         # write to 'songlist_with_lyrics.json'
#         write_to_file(data_per_song)